# Notebook 1 · Create the Main Mapper Skeleton
## 0 · Data Preparation

This notebook ingests the raw CSV (features as columns and participants as rows), standardizes the numerical columns, applies PCA, retains the minimum number of principal components that explain 95% of the variance, and uses those components in Mapper. All downstream notebooks consume the files produced here, so ensure the configuration matches your dataset before running every cell.

## 1. Environment check

Run the next cell once per session to import all functions.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import kmapper as km
from sklearn.cluster import DBSCAN
from kmapper import jupyter
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import sklearn
import matplotlib.colors as mcolors
import json
import networkx as nx
import plotly.graph_objects as go
import plotly.colors as pc
import plotly.io as pio
import ast
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import Normalize
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 2. Import Raw Data, Standardize, and Run PCA (95% Variance)

Update `df` directory and define the feature columns. This step standardizes the selected features, runs PCA, and keeps the top principal components that cumulatively explain 95% of the variance.

In [ ]:
df = pd.read_csv("Directory_to_your_data_file.csv")  # Load your data file here
feature_cols = df.columns[2:]  # Choose which columns to use as features
features = df.loc[:, feature_cols].copy()

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Keep the minimum number of PCs that explain at least 95% of variance
pca = PCA(n_components=0.95, svd_solver="full", random_state=42)
features_pca = pca.fit_transform(features_scaled)

print(f"Original feature count: {features.shape[1]}")
print(f"Selected PC count (95% variance): {features_pca.shape[1]}")
print(f"Explained variance ratio sum: {pca.explained_variance_ratio_.sum():.4f}")

## 3. Kepler Mapper

This is the main core code for the Mapper algorithm. The PCA-reduced matrix (`features_pca`) is used directly as Mapper input after standardization.

In [ ]:
mapper = km.KeplerMapper(verbose=2)

# Use first two PCs as lens when available; otherwise use all available PCs
if features_pca.shape[1] >= 2:
    projected_data = features_pca[:, :2]
else:
    projected_data = features_pca

graph = mapper.map(
    projected_data,
    X=features_pca,
    clusterer=DBSCAN(eps=5, min_samples=2),
    cover=km.Cover(n_cubes=8, perc_overlap=0.50),
)

## 4. Visualize and Save your Mapper as an HTML File

Please change `path_html` to where you want to save the HTML figure.

In [ ]:
mapper.visualize(
    graph,
    path_html=r"Y:\LabMembers\S Daneshgar\P2C\TDA\Results\3-18-2025 Sajjad\Interactive mapper\cube8-perc50\cube8-perc50.html",
)
print("Mapper graph visualization saved")

## 5. Prepare next dataframes for future use

This block creates three dataframes for you. These dataframes contain information about Mapper nodes and edges for downstream visualization and analysis.

In [ ]:
node_data = []
for node_id, sample_indices in graph["nodes"].items():
    for sample in sample_indices:
        node_data.append({"node": node_id, "sample_index": sample})
df_nodes = pd.DataFrame(node_data)

edges = []
for node1, connections in graph["links"].items():
    for node2 in connections:
        edges.append({"node_1": node1, "node_2": node2})
df_edges = pd.DataFrame(edges)
df2 = df
df_nodes['Cube'] = df_nodes['node'].str.extract(r'cube(\d+)')
df2['node_numbers'] = [[] for _ in range(len(df2))]
for _, row in df_nodes.iterrows():
    instance_idx = row.iloc[1]
    df2.at[instance_idx, 'node_numbers'] = df2.at[instance_idx, 'node_numbers'] + [row['Cube']]
df2['node_numbers'] = df2['node_numbers'].apply(lambda x: list(set(x)))
dfmapper = pd.DataFrame(df_nodes['node'].unique(), columns=['Nodes'])
dfmapper['Edges'] = [[] for _ in range(len(dfmapper))]
edges_dict = {}
for _, row in df_edges.iterrows():
    node1, node2 = row['node_1'], row['node_2']

    if node1 in edges_dict:
        edges_dict[node1].append(node2)
    else:
        edges_dict[node1] = [node2]

    if node2 in edges_dict:
        edges_dict[node2].append(node1)
    else:
        edges_dict[node2] = [node1]
dfmapper['Edges'] = dfmapper['Nodes'].map(lambda node: edges_dict.get(node, []))



## 6. Save the new dataframes

Please set the saving directory and save the new dataframes.

In [ ]:
df2.to_csv()
df_nodes.to_csv()
df_edges.to_csv()
dfmapper.to_csv()

## Continue to the next file "002_PostHoc_Mapper"